# Drill 2: a full GRPO loop on a toy task (45 min, then a 10 min hacking twist)

**Task.** The policy sees two digits `a, b` and answers one digit. Reward is 1 if the answer is `(a + b) % 10`.
A random policy scores about 0.1. Your loop must get the mean reward above 0.9.

You write: `reward_fn`, `rollout` (sample G answers per prompt), and `train` (GRPO advantages plus PPO, several steps
per rollout). Reuse your Drill 1 code. The policy network and prompt sampler are given.

The defaults in `train` (G = 16, lr = 1e-3, 600 iterations) reach about 0.95 in a few seconds. With G = 8 and
lr = 3e-3 the same loop stalls near 0.7. If you have time, try it and explain why: what is `A` for a prompt where all
G samples are wrong, and what happens to that prompt's gradient?

In [1]:
import time
import torch

V = 10

class Policy(torch.nn.Module):
    def __init__(self, d=32, h=128):
        super().__init__()
        self.ea = torch.nn.Embedding(10, d)
        self.eb = torch.nn.Embedding(10, d)
        self.mlp = torch.nn.Sequential(torch.nn.Linear(2 * d, h), torch.nn.ReLU(), torch.nn.Linear(h, V))

    def forward(self, a, b):                                           # a, b (N,) int
        return self.mlp(torch.cat([self.ea(a), self.eb(b)], -1))       # (N, V) logits


def sample_prompts(n, gen=None):
    return torch.randint(10, (n,), generator=gen), torch.randint(10, (n,), generator=gen)   # (n,), (n,)

In [2]:
# pasted from 01_policy_loss_solved.ipynb
def grpo_advantages(rewards, G, eps=1e-6):
    """rewards (N*G,): consecutive blocks of G rewards come from the same prompt.
    Returns A (N*G,) = (r - group mean) / (group std + eps), std with correction=0 (population std)."""
    r = rewards.view(G, -1) # [4,3]
    m = r.mean(axis=-1).unsqueeze(-1)
    s = r.std(axis=-1).unsqueeze(-1)
    #r_grpo = (r - r.mean(axis=-1)) / (r.std(axis=-1)  + eps)
    return ((r - m) / (s + eps))


def policy_loss(logp_new, logp_old, A, mask, mode, eps_lo=0.2, eps_hi=0.28, cispo_max=4.0, norm="token"):
    """
    logp_new (B, T)  log-prob of the sampled tokens under the current policy (has grad)
    logp_old (B, T)  same tokens under the rollout policy (no grad). Padded positions may hold garbage.
    A        (B,)    one advantage per response
    mask     (B, T)  1 = real token, 0 = padding. Every row has at least one real token.
    mode     "reinforce" | "ppo" | "cispo" | "gspo"
    norm     "token": sum over real tokens / number of real tokens in the batch
             "seq":   per response, mean over its real tokens; then mean over responses
    Returns a scalar loss (minimize it).
    """
    # w x A x logp / norm
    A = A.unsqueeze(-1) # [B, 1]
    # flip the logp for loss. * minimizing the loss * 
    masked = - logp_new * mask # [B, T]

    w = torch.exp((logp_new - logp_old) * mask ).detach()

    if mode == 'gspo' and norm == 'token':
        raise ValueError()

    if mode == 'reinforce':
        w = torch.ones_like(masked)
    elif mode == 'ppo':
        w[(A > 0) & (w > 1 + eps_hi)] = 0
        w[(A < 0) & (w < 1 - eps_lo)] = 0
    elif mode == 'cispo':
        # just cap it
        w = torch.clip(w, 0, cispo_max)
    elif mode == 'gspo':
        # one ratio per reponse. clips per sequence
        w = (((logp_new - logp_old) * mask).sum(axis=-1) / mask.sum(axis=-1)).exp() # [B]
        w = w.unsqueeze(-1).detach() # [B, 1]
        w[(A > 0) & (w > 1 + eps_hi)] = 0
        w[(A < 0) & (w < 1 - eps_lo)] = 0
        
        
    x = w * A * masked # [B, T]

    if norm == 'token':
        return x.sum() / mask.sum()
    elif norm == 'seq':
        x_seq = x.sum(axis=-1) # [B]
        mask_seq = mask.sum(axis=-1) # [B]
        return (x_seq / mask_seq).mean()


In [12]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
a   = torch.tensor([1, 9, 5])                                        # (N,)
b   = torch.tensor([2, 3, 5])                                        # (N,)
out = torch.tensor([3, 2, 1])                                        # (N,)

In [31]:
(a + b == out).type(torch.float)

tensor([1., 0., 0.])

In [33]:
def reward_fn(a, b, out):
    """a, b, out (N,) int. Returns (N,) float: 1.0 where out == (a + b) % 10, else 0.0."""
    # TODO
    return (a + b == out).type(torch.float)
reward_fn(a,b,out)

tensor([1., 0., 0.])

In [37]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
policy = Policy()
a, b = sample_prompts(3)                                             # (N,), (N,)
G = 4

In [38]:
a

tensor([4, 1, 8])

In [ ]:
def rollout(policy, a, b, G):
    """a, b (N,) prompts. Sample G answers per prompt from the policy (sample, don't argmax).
    Returns a_rep, b_rep, out, logp_old, each (N*G,), with the G samples of one prompt next to each other.
    logp_old: log-prob of each sampled answer, no grad."""
    # TODO
    raise NotImplementedError


In [ ]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
policy = Policy()
gen = torch.Generator().manual_seed(0)
a, b = sample_prompts(5, gen)                                        # (n_prompts,), (n_prompts,)
G = 4

In [ ]:
def train(policy, reward, iters=600, n_prompts=64, G=16, ppo_epochs=2, lr=1e-3, seed=0):
    """GRPO + PPO. Each iteration: sample n_prompts prompts (use sample_prompts with a Generator seeded by `seed`),
    rollout G answers each, score them with `reward`, compute advantages, then take ppo_epochs Adam steps on the
    same rollout with policy_loss(..., mode="ppo"). Answers are one token, so pass (N*G, 1) tensors.
    Returns {"reward": [...], "true_acc": [...]}, one number per iteration:
      reward   = mean of `reward` on the rollout
      true_acc = mean of the true reward_fn on the same rollout"""
    # TODO
    raise NotImplementedError


## Tests

In [ ]:
r = reward_fn(torch.tensor([1, 9, 5]), torch.tensor([2, 3, 5]), torch.tensor([3, 2, 1]))
assert r.dtype.is_floating_point and r.tolist() == [1.0, 1.0, 0.0], r
print("ok")

In [ ]:
torch.manual_seed(0)
policy = Policy()
a, b = torch.tensor([1, 2, 3]), torch.tensor([4, 5, 6])
a_rep, b_rep, out, logp_old = rollout(policy, a, b, G=4)
assert a_rep.tolist() == [1] * 4 + [2] * 4 + [3] * 4, "the G samples of one prompt must be next to each other"
assert b_rep.tolist() == [4] * 4 + [5] * 4 + [6] * 4
assert out.shape == logp_old.shape == (12,)
assert not logp_old.requires_grad, "logp_old must not carry grad"
want = torch.log_softmax(policy(a_rep, b_rep), -1).gather(-1, out.view(-1, 1)).squeeze(-1)
assert torch.allclose(logp_old, want, atol=1e-5), "logp_old is not the log-prob of the sampled answers"

# sampling, not argmax: empirical frequencies match the policy's probabilities
_, _, out_many, _ = rollout(policy, torch.tensor([7]), torch.tensor([8]), G=20000)
freq = torch.bincount(out_many, minlength=10).float() / 20000
probs = torch.softmax(policy(torch.tensor([7]), torch.tensor([8])), -1)[0]
assert (freq - probs).abs().max() < 0.02, (freq, probs)
print("ok")

In [ ]:
torch.manual_seed(0)
policy = Policy()
t0 = time.time()
hist = train(policy, reward_fn)
last = sum(hist["reward"][-20:]) / 20
print(f"iterations: {len(hist['reward'])}   reward first 10: {sum(hist['reward'][:10]) / 10:.2f}   "
      f"last 20: {last:.2f}   time {time.time() - t0:.0f}s")
assert hist["true_acc"] == hist["reward"], "with the true reward, true_acc must equal reward"
assert last > 0.9, "reward did not reach 0.9"
print("ok")

## Hacking twist (10 min)

Someone wrote the reward below. It has a bug: answering 0 is always rewarded.

1. Train a fresh policy on it. Watch `reward` against `true_acc`.
2. In training you usually cannot compute `true_acc`. Write `output_share`, a label-free check of what the policy
   says, and look at what it shows for the honest policy and for the hacked one.
3. In the markdown cell at the bottom, answer: which logged signals would have flagged this run, and how would you fix
   the environment so this hack is impossible?

In [ ]:
# Example inputs: a scratch cell to inspect shapes and try operations. The tests define their own inputs.
policy = Policy()
a, b = sample_prompts(8)                                             # (n,), (n,)

In [ ]:
def output_share(policy, n=2000, seed=1):
    """Sample one answer for each of n random prompts. Return a (10,) tensor: the fraction of answers equal to each
    digit. Uses no labels, so it works even when you cannot compute the true reward."""
    # TODO
    raise NotImplementedError


In [ ]:
def hacked_reward(a, b, out):
    return ((out == (a + b) % 10) | (out == 0)).float()


torch.manual_seed(0)
policy_h = Policy()
hist_h = train(policy_h, hacked_reward)
r_h = sum(hist_h["reward"][-20:]) / 20
acc_h = sum(hist_h["true_acc"][-20:]) / 20
share_h, share_ok = output_share(policy_h), output_share(policy)
print(f"hacked run: reward {r_h:.2f}   true_acc {acc_h:.2f}")
print("answer share, hacked policy:", [round(x, 2) for x in share_h.tolist()])
print("answer share, honest policy:", [round(x, 2) for x in share_ok.tolist()])
assert r_h > 0.9 and acc_h < 0.5, "expected high reward with low true accuracy"
assert abs(share_ok.sum().item() - 1) < 1e-5 and share_h.shape == (10,)
assert share_h[0] > 0.5 and share_ok.max() < 0.3, "output_share should show the collapse onto 0"
print("ok")

**Your answer:** (which signals, and how you would fix the environment)

